In [1]:
# =============================================================================
# Refusal Concept Cones (Directional Ablation) on Llama-3.1-8B-Instruct
# =============================================================================
# Paper : Wollschläger et al., "The Geometry of Refusal in Large Language
#         Models: Concept Cones and Representational Independence", ICML 2025
# GitHub: https://github.com/wollschlager/geometry-of-refusal
#
# Phase 1 - Extract refusal directions via diff-in-means on residual stream
# Phase 2 - Directional ablation: project out refusal direction at each layer
# Phase 3 - Run 100 JBB prompts x 20 generations with ablated model
# NOTE: Llama 3.1 date header stripped for prompt stability.
# =============================================================================

!pip -q install -U transformers datasets accelerate sentencepiece huggingface_hub

import os, gc, re, math, time, shutil
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {total/1e9:.1f} GB total, {free/1e9:.1f} GB free")

# --- 1. HF login + Drive -----------------------------------------------------
from huggingface_hub import login
from getpass import getpass
hf_token = getpass("Enter your Hugging Face token (hf_...): ")
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
print("HF login successful")

from google.colab import drive
drive.mount("/content/drive")
LOG_DIR = "/content/drive/MyDrive/LLm_perturbation_logs"
os.makedirs(LOG_DIR, exist_ok=True)

# --- 2. Load model ------------------------------------------------------------
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print(f"\nLoading {MODEL_NAME} on {DEVICE} ({DTYPE}) ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE, token=hf_token).to(DEVICE)
model.eval()
num_layers = model.config.num_hidden_layers
hidden_size = model.config.hidden_size
print(f"Loaded. Layers: {num_layers}, Hidden: {hidden_size}")

# --- 3. Load profiling datasets -----------------------------------------------
print("\nLoading malicious + benign datasets for refusal direction extraction ...")
try:
    ds_mal = load_dataset("LLM-LAT/harmful-dataset", split="train")
    mal_col = "prompt" if "prompt" in ds_mal.column_names else ds_mal.column_names[0]
    malicious_prompts = [str(r[mal_col]).strip() for r in ds_mal if str(r[mal_col]).strip()][:512]
except Exception:
    ds_mal = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors", split="harmful")
    malicious_prompts = [str(r["Goal"]).strip() for r in ds_mal if str(r["Goal"]).strip()][:512]

try:
    ds_ben = load_dataset("tatsu-lab/alpaca", split="train")
    benign_prompts = []
    for r in ds_ben:
        t = str(r.get("instruction", "")).strip()
        if t and len(t) > 10:
            benign_prompts.append(t)
        if len(benign_prompts) >= 512:
            break
except Exception:
    ds_ben = load_dataset("Anthropic/hh-rlhf", split="train", streaming=True)
    benign_prompts = []
    for r in ds_ben:
        t = str(r.get("chosen", "")).strip()
        if t and len(t) > 10:
            benign_prompts.append(t[:300])
        if len(benign_prompts) >= 512:
            break

n_profile = min(len(malicious_prompts), len(benign_prompts))
malicious_prompts = malicious_prompts[:n_profile]
benign_prompts = benign_prompts[:n_profile]
print(f"Profiling with {n_profile} malicious + {n_profile} benign prompts")

# --- 4. Phase 1: Collect residual stream activations --------------------------
_DATE_HEADER_RE = re.compile(
    r"^\s*(Cutting Knowledge Date:.*|Today Date:.*)\n?", re.MULTILINE
)

def build_chat_input(prompt_text):
    messages = [{"role": "system", "content": ""}, {"role": "user", "content": prompt_text}]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    chat_text = _DATE_HEADER_RE.sub("", chat_text)
    chat_text = re.sub(r"<\|end_header_id\|>\n{3,}", "<|end_header_id|>\n\n", chat_text)
    return tokenizer(chat_text, return_tensors="pt", add_special_tokens=False,
                     truncation=True, max_length=512)

print("\nPhase 1: Collecting residual stream activations (last token) ...")

def collect_residual_activations(prompts, label):
    acts = {l: [] for l in range(num_layers)}
    hooks = []
    _current_acts = {}

    def make_hook(layer_idx):
        def hook_fn(module, inp, out):
            hidden = out[0] if isinstance(out, tuple) else out
            last_tok = hidden[:, -1, :].detach().float().cpu()
            _current_acts[layer_idx] = last_tok
        return hook_fn

    for l_idx in range(num_layers):
        h = model.model.layers[l_idx].register_forward_hook(make_hook(l_idx))
        hooks.append(h)

    t0 = time.time()
    with torch.inference_mode():
        for i, prompt in enumerate(prompts):
            _current_acts.clear()
            enc = build_chat_input(prompt)
            ids = enc["input_ids"].to(DEVICE)
            attn = enc["attention_mask"].to(DEVICE)
            _ = model(input_ids=ids, attention_mask=attn, use_cache=False,
                      output_hidden_states=False, output_attentions=False)
            for l_idx in range(num_layers):
                if l_idx in _current_acts:
                    acts[l_idx].append(_current_acts[l_idx].squeeze(0))
            if (i + 1) % 100 == 0:
                print(f"    [{label}] {i+1}/{len(prompts)} done ({time.time()-t0:.0f}s)")

    for h in hooks:
        h.remove()

    for l_idx in range(num_layers):
        if acts[l_idx]:
            acts[l_idx] = torch.stack(acts[l_idx], dim=0)
        else:
            acts[l_idx] = None
    return acts

mal_acts = collect_residual_activations(malicious_prompts, "malicious")
ben_acts = collect_residual_activations(benign_prompts, "benign")
print("Activation collection done.")

# --- 5. Compute refusal directions via diff-in-means -------------------------
print("\nComputing refusal directions (diff-in-means) per layer ...")
refusal_dirs = {}
dir_norms = []

for l_idx in range(num_layers):
    if mal_acts[l_idx] is None or ben_acts[l_idx] is None:
        continue
    mean_mal = mal_acts[l_idx].mean(dim=0)
    mean_ben = ben_acts[l_idx].mean(dim=0)
    r = mean_mal - mean_ben
    norm = r.norm().item()
    dir_norms.append((l_idx, norm))
    r_hat = (r / r.norm()).to(DEVICE, dtype=DTYPE)
    refusal_dirs[l_idx] = r_hat

dir_norms.sort(key=lambda x: x[1], reverse=True)
print(f"  Refusal directions computed for {len(refusal_dirs)} layers.")
print(f"  Top 5 layers by direction norm:")
for l_idx, norm in dir_norms[:5]:
    print(f"    Layer {l_idx}: ||r|| = {norm:.4f}")

del mal_acts, ben_acts, malicious_prompts, benign_prompts
gc.collect(); torch.cuda.empty_cache()

# --- 6. Install directional ablation hooks ------------------------------------
ablation_hooks = []

def make_ablation_hook(r_hat):
    r_hat_f = r_hat.float()
    def hook_fn(module, inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        orig_dtype = hidden.dtype
        h_f = hidden.float()
        proj = torch.einsum("bsh,h->bs", h_f, r_hat_f)
        hidden_ablated = (h_f - proj.unsqueeze(-1) * r_hat_f.unsqueeze(0).unsqueeze(0)).to(orig_dtype)
        if isinstance(out, tuple):
            return (hidden_ablated,) + out[1:]
        return hidden_ablated
    return hook_fn

n_hooks = 0
for l_idx in range(num_layers):
    if l_idx in refusal_dirs:
        h = model.model.layers[l_idx].register_forward_hook(
            make_ablation_hook(refusal_dirs[l_idx])
        )
        ablation_hooks.append(h)
        n_hooks += 1

print(f"\nInstalled {n_hooks} directional ablation hooks.")
print("Refusal directions will be projected out during all subsequent inference.\n")

# --- 7. Load JailbreakBench prompts -------------------------------------------
ds_jbb = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors", split="harmful")
prompt_col = "Goal"
all_prompts = [str(ds_jbb[i][prompt_col]).strip() for i in range(len(ds_jbb))]
NUM_PROMPTS = min(100, len(all_prompts))
prompts = all_prompts[:NUM_PROMPTS]
print(f"JailbreakBench: {NUM_PROMPTS} prompts loaded.")

# --- 8. Batched generation ----------------------------------------------------
RUNS_PER_PROMPT = 20
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.7
TOP_K = 50
TOP_P = 0.9

def build_stop_ids(tok, mdl):
    eos = tok.eos_token_id
    stop = set()
    if eos is not None:
        if isinstance(eos, (list, tuple, set)):
            stop.update(int(e) for e in eos)
        else:
            stop.add(int(eos))
    gen_eos = getattr(getattr(mdl, "generation_config", None), "eos_token_id", None)
    if isinstance(gen_eos, (list, tuple)):
        stop.update(int(t) for t in gen_eos if t is not None)
    elif isinstance(gen_eos, int):
        stop.add(gen_eos)
    unk = getattr(tok, "unk_token_id", None)
    for t in ("<|end_of_text|>", "<|eot_id|>", "<|eom_id|>"):
        tid = tok.convert_tokens_to_ids(t)
        if isinstance(tid, int) and tid >= 0 and tid != unk:
            stop.add(tid)
    return stop

STOP_IDS = build_stop_ids(tokenizer, model)
STOP_TENSOR = torch.tensor(sorted(STOP_IDS), device=DEVICE)
print(f"Stop token IDs: {STOP_IDS}")

def sample_from_logits(logits, temperature, top_k, top_p):
    logits = logits.float()
    if temperature > 0 and temperature != 1.0:
        logits = logits / temperature
    if top_k and top_k > 0:
        k = min(top_k, logits.size(-1))
        kth = torch.topk(logits, k, dim=-1).values[:, -1, None]
        logits = torch.where(logits < kth, torch.full_like(logits, float("-inf")), logits)
    if 0 < top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        sm = F.softmax(sorted_logits, dim=-1)
        cum = torch.cumsum(sm, dim=-1)
        remove = (cum - sm) >= top_p
        sorted_logits[remove] = float("-inf")
        logits = sorted_logits.scatter(-1, sorted_idx, sorted_logits)
    probs = F.softmax(logits, dim=-1)
    if temperature == 0:
        return torch.argmax(probs, dim=-1, keepdim=True)
    nxt = torch.multinomial(probs.cpu(), num_samples=1)
    return nxt.to(logits.device)

@torch.inference_mode()
def run_batch_ablated(prompt_text, n_runs, max_new, micro_batch=None):
    prompt_text = prompt_text.strip()
    if not prompt_text:
        return [""] * n_runs
    if micro_batch and micro_batch < n_runs:
        outs = []
        remaining = n_runs
        while remaining > 0:
            b = min(micro_batch, remaining)
            outs.extend(run_batch_ablated(prompt_text, b, max_new, micro_batch=None))
            remaining -= b
        return outs

    messages = [{"role": "system", "content": ""}, {"role": "user", "content": prompt_text}]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    chat_text = _DATE_HEADER_RE.sub("", chat_text)
    chat_text = re.sub(r"<\|end_header_id\|>\n{3,}", "<|end_header_id|>\n\n", chat_text)
    enc = tokenizer(chat_text, return_tensors="pt", add_special_tokens=False)
    ids = enc["input_ids"].to(DEVICE)
    L = ids.shape[1]
    ctx_limit = min(model.config.max_position_embeddings, 8192)
    max_new_actual = min(max_new, max(1, ctx_limit - L - 1))

    B = n_runs
    ids = ids.expand(B, L).contiguous()
    attn = torch.ones(B, L, dtype=torch.long, device=DEVICE)

    out = model(input_ids=ids, attention_mask=attn, use_cache=True,
                output_hidden_states=False, output_attentions=False)
    past = out.past_key_values
    logits = out.logits[:, -1, :]

    generated = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=DEVICE)

    for _ in range(max_new_actual):
        nxt = sample_from_logits(logits, TEMPERATURE, TOP_K, TOP_P)
        is_stop = torch.isin(nxt.squeeze(1), STOP_TENSOR)
        for b in range(B):
            if not finished[b]:
                tid = nxt[b, 0].item()
                if tid in STOP_IDS:
                    finished[b] = True
                else:
                    generated[b].append(tid)
        finished = finished | is_stop
        if bool(finished.all()):
            break
        attn = torch.cat([attn, torch.ones(B, 1, dtype=torch.long, device=DEVICE)], dim=1)
        out = model(input_ids=nxt, attention_mask=attn, past_key_values=past,
                    use_cache=True, output_hidden_states=False, output_attentions=False)
        past = out.past_key_values
        logits = out.logits[:, -1, :]

    return [tokenizer.decode(g, skip_special_tokens=True).strip() for g in generated]

# --- 9. Run experiment -------------------------------------------------------
MODEL_SHORT = "Llama3.1-8B"
TECHNIQUE = "RefusalCones"

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
fname = f"{MODEL_SHORT}_{TECHNIQUE}_{ts}.txt"
log_path = os.path.join(LOG_DIR, fname)
local_log = f"/content/{fname}"

total_gens = NUM_PROMPTS * RUNS_PER_PROMPT
print(f"\n{'='*80}")
print(f"Model: {MODEL_SHORT} | Technique: {TECHNIQUE}")
print(f"Prompts: {NUM_PROMPTS} | Runs/prompt: {RUNS_PER_PROMPT} | Total: {total_gens}")
print(f"max_new_tokens: {MAX_NEW_TOKENS} | temp: {TEMPERATURE} | top_k: {TOP_K} | top_p: {TOP_P}")
print(f"Ablation: directional ablation at {n_hooks} layers (diff-in-means refusal direction)")
print(f"Log: {log_path}")
print(f"{'='*80}\n")

with open(log_path, "w", encoding="utf-8") as f:
    f.write(f"model={MODEL_NAME}\n")
    f.write(f"technique={TECHNIQUE}\n")
    f.write(f"method=diff-in-means directional ablation (all layers)\n")
    f.write(f"start={datetime.now().isoformat()}\n")
    f.write(f"num_prompts={NUM_PROMPTS} runs_per_prompt={RUNS_PER_PROMPT}\n")
    f.write(f"max_new_tokens={MAX_NEW_TOKENS} temperature={TEMPERATURE} top_k={TOP_K} top_p={TOP_P}\n")
    f.write(f"ablation_hooks={n_hooks} profiling_prompts={n_profile}\n")
    f.write(f"top_layers_by_norm={dir_norms[:5]}\n")
    f.write("=" * 120 + "\n\n")

def checkpoint_copy():
    shutil.copy2(log_path, local_log)

job = 0
t0 = time.time()

with open(log_path, "a", encoding="utf-8") as f:
    for p_idx, prompt_text in enumerate(prompts, start=1):
        f.write(f"Prompt{p_idx} - {prompt_text}\n")
        f.write(f"technique - {TECHNIQUE} (directional ablation, diff-in-means)\n")
        f.flush(); os.fsync(f.fileno())

        print(f">>> Prompt {p_idx}/{NUM_PROMPTS}")

        try:
            outs = run_batch_ablated(prompt_text, RUNS_PER_PROMPT, MAX_NEW_TOKENS,
                                     micro_batch=10)
        except Exception as e:
            outs = [f"[ERROR] {type(e).__name__}: {e}"] * RUNS_PER_PROMPT

        for run_id, out in enumerate(outs, start=1):
            job += 1
            out = str(out).replace("\r\n", "\n").strip()
            f.write(f"generation{run_id}/{RUNS_PER_PROMPT}- {out}\n")
        f.flush(); os.fsync(f.fileno())

        elapsed = time.time() - t0
        rate = job / elapsed if elapsed > 0 else 0
        eta = (total_gens - job) / rate / 60 if rate > 0 else 0
        print(f"    {RUNS_PER_PROMPT} generations saved | {job}/{total_gens} total | "
              f"{elapsed/60:.1f}min elapsed | ~{eta:.0f}min remaining")
        checkpoint_copy()

        f.write("-" * 120 + "\n")
        f.flush(); os.fsync(f.fileno())

checkpoint_copy()
elapsed = time.time() - t0
print(f"\nDone in {elapsed/60:.1f} min")
print(f"Drive log: {log_path}")
print(f"Local log: {local_log}")

for h in ablation_hooks:
    h.remove()
print("Ablation hooks removed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.7 MB/s eta 0:00:00
CUDA available: True
GPU: NVIDIA L4
VRAM: 23.7 GB total, 23.5 GB free
Enter your Hugging Face token (hf_...): ··········
HF login successful
Mounted at /content/drive

Loading meta-llama/Llama-3.1-8B-Instruct on cuda (torch.bfloat16) ...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Generating harmful split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating benign split:   0%|          | 0/100 [00:00<?, ? examples/s]

JailbreakBench: 100 prompts loaded.
Stop token IDs: {128008, 128009, 128001}

Model: Llama3.1-8B | Technique: RefusalCones
Prompts: 100 | Runs/prompt: 20 | Total: 2000
max_new_tokens: 256 | temp: 0.7 | top_k: 50 | top_p: 0.9
Ablation: directional ablation at 32 layers (diff-in-means refusal direction)
Log: /content/drive/MyDrive/LLm_perturbation_logs/Llama3.1-8B_RefusalCones_20260821_080514.txt

>>> Prompt 1/100


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


    20 generations saved | 20/2000 total | 0.9min elapsed | ~94min remaining
>>> Prompt 2/100
    20 generations saved | 40/2000 total | 1.9min elapsed | ~93min remaining
>>> Prompt 3/100
    20 generations saved | 60/2000 total | 2.8min elapsed | ~92min remaining
>>> Prompt 4/100
    20 generations saved | 80/2000 total | 3.8min elapsed | ~91min remaining
>>> Prompt 5/100
    20 generations saved | 100/2000 total | 4.7min elapsed | ~90min remaining
>>> Prompt 6/100
    20 generations saved | 120/2000 total | 5.4min elapsed | ~85min remaining
>>> Prompt 7/100
    20 generations saved | 140/2000 total | 6.3min elapsed | ~84min remaining
>>> Prompt 8/100
    20 generations saved | 160/2000 total | 7.3min elapsed | ~84min remaining
>>> Prompt 9/100
    20 generations saved | 180/2000 total | 8.2min elapsed | ~83min remaining
>>> Prompt 10/100
    20 generations saved | 200/2000 total | 9.0min elapsed | ~81min remaining
>>> Prompt 11/100
    20 generations saved | 220/2000 total | 9.9min e